In [1]:
import numpy as np
import os
from PIL import Image

In [2]:
@staticmethod
def FindDepthMinMaxDifferences(pairs):
    differences = []

    for paths, low, high in pairs:
        depth_hr_path, rgb_path, depth_lr_path = paths

        # Load depths in meters
        gt_depth = np.array(Image.open(depth_hr_path)).astype(np.float32) / 1000.0
        lr_depth = np.array(Image.open(depth_lr_path)).astype(np.float32) / 1000.0

        # Clip using the same range used during processing
        gt_depth = np.clip(gt_depth, low, high)
        lr_depth = np.clip(lr_depth, low, high)

        # Calculate min/max independently
        gt_min = gt_depth.min()
        gt_max = gt_depth.max()

        lr_min = lr_depth.min()
        lr_max = lr_depth.max()

        # Store if LR and GT min/max are different
        if gt_min != lr_min or gt_max != lr_max:
            differences.append({
                "gt_path": depth_hr_path,
                "lr_path": depth_lr_path,
                "gt_min": gt_min,
                "gt_max": gt_max,
                "lr_min": lr_min,
                "lr_max": lr_max
            })

    return differences

In [3]:
from Utilities.PathManager import PathManager
from Utilities.ProcessingRGBDDReal import ProcessingRGBDDReal

# 1. Init example paths
model_train = PathManager.GetBasePath() + 'RGBDD-Full/models/models_train'
model_test = PathManager.GetBasePath() + 'RGBDD-Full/models/models_test'
plants_train = PathManager.GetBasePath() + 'RGBDD-Full/plants/plants_train'
plants_test = PathManager.GetBasePath() + 'RGBDD-Full/plants/plants_test'
portraits_train = PathManager.GetBasePath() + 'RGBDD-Full/portraits/portraits_train'
portraits_test = PathManager.GetBasePath() + 'RGBDD-Full/portraits/portraits_test'

model_train_pairs = ProcessingRGBDDReal._LoadPairPaths(model_train)
model_test_pairs = ProcessingRGBDDReal._LoadPairPaths(model_test)


./RGBDD-Full/models/models_train
./RGBDD-Full/models/models_test


In [4]:
plants_train_pairs = ProcessingRGBDDReal._LoadPairPaths(plants_train)
plants_test_pairs = ProcessingRGBDDReal._LoadPairPaths(plants_test)

./RGBDD-Full/plants/plants_train
./RGBDD-Full/plants/plants_test


In [5]:
portraits_train_pairs = ProcessingRGBDDReal._LoadPairPaths(portraits_train)
portraits_test_pairs = ProcessingRGBDDReal._LoadPairPaths(portraits_test)

train_pairs = []
test_pairs = []
for i, low, high in [(model_train_pairs, 0.6, 3), (portraits_train_pairs, 1, 5), (plants_train_pairs, 0.6, 1.5)]:
    for v in i.values():
        train_pairs.append(
            (
                v, low, high
            )
        )

# 4. Merge testing examples
for i, low, high in [(model_test_pairs, 0.6, 3), (portraits_test_pairs, 1, 5), (plants_test_pairs, 0.6, 1.5)]:
    for v in i.values():
        test_pairs.append(
            (
                v, low, high
            )
        )


train_differences = FindDepthMinMaxDifferences(train_pairs)
test_differences = FindDepthMinMaxDifferences(test_pairs)

print("Train samples with different min/max:", len(train_differences))
print("Test samples with different min/max:", len(test_differences))

for item in train_differences[:10]:
    print(item)

./RGBDD-Full/portraits/portraits_train
./RGBDD-Full/portraits/portraits_test
Train samples with different min/max: 2200
Test samples with different min/max: 403
{'gt_path': './RGBDD-Full/models/models_train/20200719111616/20200719111616_HR_gt.png', 'lr_path': './RGBDD-Full/models/models_train/20200719111616/20200719111616_LR_fill_depth.png', 'gt_min': np.float32(1.425), 'gt_max': np.float32(2.598), 'lr_min': np.float32(1.429), 'lr_max': np.float32(2.57)}
{'gt_path': './RGBDD-Full/models/models_train/20200715091505/20200715091505_HR_gt.png', 'lr_path': './RGBDD-Full/models/models_train/20200715091505/20200715091505_LR_fill_depth.png', 'gt_min': np.float32(1.369), 'gt_max': np.float32(3.0), 'lr_min': np.float32(1.563), 'lr_max': np.float32(3.0)}
{'gt_path': './RGBDD-Full/models/models_train/20200715095734/20200715095734_HR_gt.png', 'lr_path': './RGBDD-Full/models/models_train/20200715095734/20200715095734_LR_fill_depth.png', 'gt_min': np.float32(0.6), 'gt_max': np.float32(3.0), 'lr_min':

In [6]:
diff_min = []
diff_max = []

for item in train_differences:
    diff_min.append(abs(item['gt_min'] - item['lr_min']))
    diff_max.append(abs(item['gt_max'] - item['lr_max']))

print(sum(diff_max) / len(diff_max))
print(sum(diff_min) / len(diff_min))

0.073865466
0.068281874
